In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import math
import random

import tqdm.notebook
from tqdm.notebook import trange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## MCTS Random Walk

In [ ]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken = None):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken

        self.children = []
        self.expandable_moves = game.get_legal_moves(state)

        self.visit_count = 0
        self.value_sum = 0

    def is_fully_expanded(self):
        return np.sum(self.expandable_moves) == 0 and len(self.children) > 0


    def select(self):
        best_child = None
        best_ucb = -np.inf

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child

        return best_child


    def get_ucb(self, child):
        q = 1 - ((child.value_sum / child.visit_count) + 1) / 2     # child is opposite player so bad position preferred
        
        return q + self.args['C'] * math.sqrt(math.log(self.visit_count)/child.visit_count)



    def expand(self):
        action = np.random.choice(np.where(self.expandable_moves == 1)[0]) # choose a legal move
        self.expandable_moves[action] = 0

        child_state = self.state.copy()
        child_state = self.game.get_next_state(child_state, action, 1)
        child_state = self.game.change_perspective(child_state, player = -1) # flips pov frmo -1 -> 1

        child = Node(self.game, self.args, child_state, self, action)
        self.children.append(child)
        return child


    def simulate(self):
        value, terminated = self.game.get_val_and_terminated(self.state, self.action_taken)
        value = self.game.get_opponent_value(value)

        if terminated:
            return value

        #perform rollout

        rollout_state = self.state.copy()
        rollout_player = 1

        while True:
            valid_moves = self.game.get_legal_moves(rollout_state)
            action = np.random.choice(np.where(valid_moves == 1)[0])
            rollout_state = self.game.get_next_state(rollout_state, action, rollout_player)
            
            value, terminated = self.game.get_val_and_terminated(rollout_state, action)
            if terminated:
                if rollout_player == -1:
                    value = self.game.get_opponent_value(value)
                return value

            rollout_player = self.game.get_opponent(rollout_player)


    def backprop(self, value):
        self.value_sum += value
        self.visit_count += 1

        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backprop(value)



class MCTSrandom:
    def __init__(self, game, args):
        self.game = game
        self.args = args

    def search(self, state):
        root = Node(self.game, self.args, state)
        
        for search in range(self.args['num_searches']):
            node = root

            while node.is_fully_expanded():
                node = node.select() # selection

            value, terminated = self.game.get_val_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not terminated: # leaf node + non terminated game = expand + simulate
                node = node.expand() #expansion
                value = node.simulate() # simulate

            node.backprop(value) #backprop
        
        action_prob = np.zeros(self.game.action_sz)
        for child in root.children:
            action_prob[child.action_taken] = child.visit_count

        action_prob /= np.sum(action_prob)
        return action_prob

## MCTS with Value Model

In [ ]:
class Node:
    def __init__(self, game, args, state, parent=None, action_taken = None, prior = 0, visit_count = 0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = prior # prior added

        self.children = []

        self.visit_count = visit_count
        self.value_sum = 0

    def is_fully_expanded(self):
        return len(self.children) > 0


    def select(self):
        best_child = None
        best_ucb = -np.inf

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child

        return best_child


    def get_ucb(self, child): #updated ucb
        if child.visit_count == 0:
            q = 0
        else:
            q = 1 - ((child.value_sum / child.visit_count) + 1)/2

        return q + self.args['C'] * child.prior * math.sqrt(self.visit_count) / (1 + child.visit_count)



    def expand(self, policy):

        for action, prob in enumerate(policy):
            if prob > 0:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action, 1)
                child_state = self.game.change_perspective(child_state, player = -1) # flips pov frmo -1 -> 1
                child = Node(self.game, self.args, child_state, self, action, prob)
                self.children.append(child)



    def backprop(self, value):
        self.value_sum += value
        self.visit_count += 1

        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backprop(value)



class MCTS:
    def __init__(self, game, args, model):
        self.game = game
        self.args = args
        self.model = model # added model 


    @torch.no_grad() # no need for gradients during mcts
    def search(self, state):
        root = Node(self.game, self.args, state, visit_count = 1)

        policy, value = self.model(torch.tensor(self.game.get_encoded_state(state), device=device).unsqueeze(0))
        policy = torch.softmax(policy, axis=-1).squeeze(0).cpu().numpy()

        #add drichlet noise for exploration
        policy = (1 - self.args['epsilon']) * policy + \
            self.args['epsilon'] * np.random.dirichlet([self.args['dirichlet_eta'] * self.game.action_sz])
        
        valid_moves = self.game.get_legal_moves(state)
        policy = policy * valid_moves
        policy = policy / np.sum(policy)
        root.expand(policy)
        
        for search in range(self.args['num_searches']):
            node = root

            while node.is_fully_expanded():
                node = node.select() # selection

            value, terminated = self.game.get_val_and_terminated(node.state, node.action_taken)
            value = self.game.get_opponent_value(value)

            if not terminated: # leaf node + non terminated game = expand + simulate

                #integrated model to mcts
                encoded = torch.tensor(self.game.get_encoded_state(node.state), device=self.model.device).unsqueeze(0) # input on same device as model
                policy, value = self.model(encoded)
                
                policy = torch.softmax(policy, dim = 1).squeeze(0).cpu().numpy() # back to cpu for numpy
                valid_moves = self.game.get_legal_moves(node.state)
                policy = policy*valid_moves #mask
                policy /= np.sum(policy)

                value = value.item() # gets flota from tensor

                node.expand(policy) #expansion


            node.backprop(value) #backprop != optimizer.backprop()
        
        action_prob = np.zeros(self.game.action_sz)
        for child in root.children:
            action_prob[child.action_taken] = child.visit_count

        action_prob /= np.sum(action_prob)
        return action_prob